# EDA Notebook 1 — LMS Events Bronze
**Source**: `mini_project_grp6.bronze.lms_events_bronze`  
**Format**: NDJSON | ~3M events/day | 6 event types  
**Goal**: Understand event distribution, nulls, time coverage, device/country spread

In [0]:
# ============================================================
# CELL 2 — Setup
# ============================================================
# Imports and Spark config for EDA session

from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd

# spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

CATALOG  = "mini_project_grp6"
BRONZE   = "bronze"
TABLE    = f"{CATALOG}.{BRONZE}.lms_events_bronze"

print(f"Reading from: {TABLE}")

In [0]:
# ============================================================
# CELL 3 — Load & Schema
# ============================================================
# Load bronze table and inspect schema

df = spark.table(TABLE)

print(f"Schema:")
df.printSchema()

In [0]:
# ============================================================
# CELL 4 — Row Count & Partition Coverage
# ============================================================
# How many rows total? How many distinct event dates?

total_rows = df.count()
distinct_dates = df.select("event_date").distinct().count()

print(f"Total rows       : {total_rows:,}")
print(f"Distinct dates   : {distinct_dates}")
print(f"\nDate range:")
df.selectExpr(
    "min(event_date) as earliest_date",
    "max(event_date) as latest_date"
).show(truncate=False)

In [0]:
# ============================================================
# CELL 5 — Null / Missing Value Audit
# ============================================================
# Check every column for nulls — critical for DQ baseline

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_pd = null_counts.toPandas().T.reset_index()
null_pd.columns = ["column", "null_count"]
null_pd["null_pct"] = (null_pd["null_count"] / total_rows * 100).round(2)
null_pd = null_pd.sort_values("null_count", ascending=False)

print("=== NULL AUDIT ===")
print(null_pd.to_string(index=False))

In [0]:
# ============================================================
# CELL 6 — Event Type Distribution
# ============================================================
# Which event types dominate? Are all 6 types present?

print("=== EVENT TYPE DISTRIBUTION ===")
event_dist = (
    df.groupBy("event_type")
      .agg(
          F.count("*").alias("count"),
          F.round(F.count("*") / total_rows * 100, 2).alias("pct")
      )
      .orderBy("count", ascending=False)
)
display(event_dist)

In [0]:
# ============================================================
# CELL 7 — Future-Dated Event Check (DQ Rule)
# ============================================================
# Business rule: event_ts must NOT be future-dated

from datetime import datetime

current_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

future_events = df.filter(F.col("event_ts") > F.lit(current_ts))
future_count  = future_events.count()

print(f"Future-dated events : {future_count:,}")
print(f"Run timestamp       : {current_ts}")

if future_count > 0:
    print("\nSample future-dated records:")
    display(future_events.select("event_id", "student_id", "event_ts").limit(10))
else:
    print("✅ No future-dated events found")

In [0]:
# ============================================================
# CELL 8 — Student & Course Coverage
# ============================================================
# How many unique students and courses appear in events?

coverage = df.agg(
    F.countDistinct("student_id").alias("unique_students"),
    F.countDistinct("course_id").alias("unique_courses"),
    F.countDistinct("module_id").alias("unique_modules"),
    F.sum(F.col("student_id").isNull().cast("int")).alias("null_student_ids"),
    F.sum(F.col("course_id").isNull().cast("int")).alias("null_course_ids")
)
display(coverage)

In [0]:
# ============================================================
# CELL 9 — Duration Distribution
# ============================================================
# Understand session duration_seconds — spot anomalies

df.select("duration_seconds").describe().show()

print("\nPercentile breakdown (duration_seconds):")
df.selectExpr(
    "percentile_approx(duration_seconds, 0.25) as p25",
    "percentile_approx(duration_seconds, 0.50) as p50",
    "percentile_approx(duration_seconds, 0.75) as p75",
    "percentile_approx(duration_seconds, 0.95) as p95",
    "percentile_approx(duration_seconds, 0.99) as p99",
    "max(duration_seconds) as max_val"
).show()

In [0]:
# ============================================================
# CELL 10 — Device Type & Country Distribution
# ============================================================
# Who is using the platform and from where?

print("=== DEVICE TYPE ===")
display(
    df.groupBy("device_type")
      .count()
      .orderBy("count", ascending=False)
)

print("=== TOP 10 COUNTRIES ===")
display(
    df.groupBy("ip_country")
      .count()
      .orderBy("count", ascending=False)
      .limit(10)
)

In [0]:
# ============================================================
# CELL 11 — Daily Event Volume Trend
# ============================================================
# Are events evenly distributed across 6 months?

display(
    df.groupBy("event_date")
      .agg(F.count("*").alias("event_count"))
      .orderBy("event_date")
)

In [0]:
# ============================================================
# CELL 12 — Metadata Column Check
# ============================================================
# Confirm all Bronze metadata columns exist and are non-null

for col in ["_source_file", "_load_ts", "_schema_version"]:
    null_c = df.filter(F.col(col).isNull()).count()
    print(f"{col}: {null_c} nulls")

In [0]:
# ============================================================
# CELL 13 — EDA Summary
# ============================================================

print("=" * 55)
print("LMS EVENTS BRONZE — EDA SUMMARY")
print("=" * 55)
print(f"  Total rows           : {total_rows:,}")
print(f"  Date coverage        : {distinct_dates} days")
print(f"  Unique students      : (see Cell 8)")
print(f"  Unique courses       : (see Cell 8)")
print(f"  Future-dated events  : {future_count:,}")
print(f"  Event types present  : (see Cell 6)")
print("=" * 55)